In [2]:
import sqlite3
import pandas as pd
from pathlib import Path

In [3]:
db_path = Path("../data/ecommerce.db")

conn = sqlite3.connect(db_path)

In [4]:
def run_sql(query):
    return pd.read_sql(query, conn)

In [5]:
conn.executescript("""
DROP VIEW IF EXISTS vw_fact_sales;

CREATE VIEW vw_fact_sales AS

WITH order_totals AS (

    SELECT

        order_id,

        SUM(price) AS total_merchandise_value

    FROM order_items

    GROUP BY order_id

)

SELECT

    oi.order_id,

    oi.order_item_id,

    c.customer_unique_id,

    oi.product_id,

    oi.seller_id,

    DATE(o.order_purchase_timestamp) AS purchase_date,

    o.order_status,

    oi.price,

    oi.freight_value,

    (oi.price + oi.freight_value) AS gross_value,

    ROUND(

        (oi.price * 1.0 / ot.total_merchandise_value)

        *

        vco.total_payment,

        2

    ) AS allocated_payment,

    r.review_score,

    ROUND(

        JULIANDAY(o.order_delivered_customer_date)

        -

        JULIANDAY(o.order_purchase_timestamp),

        2

    ) AS delivery_days

FROM order_items oi

JOIN orders o
ON oi.order_id = o.order_id

JOIN customers c
ON o.customer_id = c.customer_id

JOIN vw_customer_orders vco
ON oi.order_id = vco.order_id

JOIN order_totals ot
ON oi.order_id = ot.order_id

LEFT JOIN reviews r
ON oi.order_id = r.order_id;

""")

conn.commit()

print("✅ vw_fact_sales created successfully")

✅ vw_fact_sales created successfully


In [6]:
run_sql("""
SELECT *
FROM vw_fact_sales
LIMIT 5;
""")

,order_id,order_item_id,customer_unique_id,product_id,seller_id,purchase_date,order_status,price,freight_value,gross_value,allocated_payment,review_score,delivery_days
0,00010242fe8c5a6d1ba2dd792cb16214,1,871766c5855e863f6eccc05f988b23cb,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-13,delivered,58.90,13.29,72.19,72.19,5,7.61
1,00018f77f2f0320c557190d7a144bdd3,1,eb28e67c4c0b83846050ddfb8a35d051,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-04-26,delivered,239.90,19.93,259.83,259.83,4,16.22
2,000229ec398224ef6ca0657da4fc703e,1,3818d81c6709e39d06b2738a8d3a2474,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-14,delivered,199.00,17.87,216.87,216.87,5,7.95
3,00024acbcdf0a6daa1e931b038114c75,1,af861d436cfc08b2c2ddefd0ba074622,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-08,delivered,12.99,12.79,25.78,25.78,4,6.15
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,64b576fb70d441e8f1b2d7d446e483c5,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-04,delivered,199.90,18.14,218.04,218.04,5,25.11


In [7]:
run_sql("""
SELECT name
FROM sqlite_master
WHERE type='view';
""")

,name
0,vw_orders
1,vw_order_items
2,vw_customer_orders
3,vw_product_sales
4,vw_fact_sales


In [8]:
run_sql("""
SELECT *
FROM vw_fact_sales
LIMIT 5;
""")

,order_id,order_item_id,customer_unique_id,product_id,seller_id,purchase_date,order_status,price,freight_value,gross_value,allocated_payment,review_score,delivery_days
0,00010242fe8c5a6d1ba2dd792cb16214,1,871766c5855e863f6eccc05f988b23cb,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-13,delivered,58.90,13.29,72.19,72.19,5,7.61
1,00018f77f2f0320c557190d7a144bdd3,1,eb28e67c4c0b83846050ddfb8a35d051,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-04-26,delivered,239.90,19.93,259.83,259.83,4,16.22
2,000229ec398224ef6ca0657da4fc703e,1,3818d81c6709e39d06b2738a8d3a2474,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-14,delivered,199.00,17.87,216.87,216.87,5,7.95
3,00024acbcdf0a6daa1e931b038114c75,1,af861d436cfc08b2c2ddefd0ba074622,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-08,delivered,12.99,12.79,25.78,25.78,4,6.15
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,64b576fb70d441e8f1b2d7d446e483c5,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-04,delivered,199.90,18.14,218.04,218.04,5,25.11


In [9]:
run_sql("""
SELECT COUNT(*)
FROM order_items;
""")

,COUNT(*)
0,112650


In [10]:
run_sql("""
SELECT COUNT(*)
FROM vw_customer_orders;
""")

,COUNT(*)
0,99441


In [11]:
conn.executescript("""

DROP VIEW IF EXISTS vw_dim_customer;

CREATE VIEW vw_dim_customer AS

SELECT

    c.customer_unique_id,

    MIN(o.order_purchase_timestamp) AS first_purchase_date,

    MAX(c.customer_city) AS customer_city,

    MAX(c.customer_state) AS customer_state,

    COUNT(DISTINCT o.order_id) AS total_orders

FROM customers c

JOIN orders o

ON c.customer_id = o.customer_id

GROUP BY

    c.customer_unique_id;

""")

conn.commit()

print("✅ vw_dim_customer created successfully")

✅ vw_dim_customer created successfully


In [12]:
run_sql("""

SELECT *

FROM vw_dim_customer

LIMIT 5;

""")

,customer_unique_id,first_purchase_date,customer_city,customer_state,total_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,cajamar,SP,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,osasco,SP,1
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,sao jose,SC,1
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,belem,PA,1
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,sorocaba,SP,1


In [13]:
run_sql("""

SELECT COUNT(*)

FROM vw_dim_customer;

""")

,COUNT(*)
0,96096


In [14]:
run_sql("""

SELECT

COUNT(*) AS total_rows,

COUNT(DISTINCT customer_unique_id) AS unique_customers

FROM vw_dim_customer;

""")

,total_rows,unique_customers
0,96096,96096
